In [ ]:
import json
import sys
from rich import print as rprint
from rich import inspect
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))


from scripts.text_matching import normalise_text

In [17]:
people_file = Path(project_root / "data/from db/people.json")

In [18]:
with open(people_file, "r") as f:
   people = json.load(f)


In [19]:
multiples_dict = {}
people_minus_multiples = []

SEPARATORS = re.compile(r"\s*;\s*|\s+/\s+|\s+und\s+")
ETC_SUFFIX = re.compile(r"\s+u\.a\.\s*$")

for person in people:
    first = person.get("given_names", "")
    id = person["person_id"]

    if not first:
        people_minus_multiples.append(person)
        continue

    if SEPARATORS.search(first):
        multiples_dict[id] = person
    elif ETC_SUFFIX.search(first):
        multiples_dict[id] = person
    else:
        people_minus_multiples.append(person)

rprint(f"total people:{len(people)}")

rprint(f"total multiples: {len(multiples_dict)}")
rprint(f"minus_multiples: {len(people_minus_multiples)}")
rprint(dict(list(multiples_dict.items())[:3]))


total people:7807

total multiples: 18

minus_multiples: 7789

{
    158: {
        'person_id': 158,
        'unified_id': 'arnim_bettina_und_gisela_von',
        'family_name': 'Arnim',
        'given_names': 'Bettina und Gisela',
        'name_particles': 'von',
        'single_name': None,
        'is_organisation': False,
        'created_at': '2026-01-26 19:54:23.311152',
        'updated_at': '2026-01-26 19:54:23.311152'
    },
    532: {
        'person_id': 532,
        'unified_id': 'bettermann_rainer_r',
        'family_name': 'Bettermann',
        'given_names': 'Rainer und Rosi',
        'name_particles': None,
        'single_name': None,
        'is_organisation': False,
        'created_at': '2026-01-26 19:54:23.311152',
        'updated_at': '2026-01-26 19:54:23.311152'
    },
    696: {
        'person_id': 696,
        'unified_id': 'bontjes_van_bee_roseli_s',
        'family_name': 'Bontjes Van Bee',
        'given_names': 'Roseli und Saskia',
        'name_particles': None,
        'single_name': None,
        'is_organisation': False,
        'created_at': '2026-01-26 19:54:23.311152',
        'updated_at': '2026-01-26 19:54:23.311152'
    }
}


    UNIFIED_ID GENERATION:
    - Format: lowercase, underscore-separated
    - Use: family_name + given_names (first name + middle initial if present)
    - Example: "Adorno, Theodor W." → "adorno_theodor_w"
    - Example: "MOZART, Wolfgang Amadeus" → "mozart_wolfgang_a"
    - For organisations or single names → use normalized form
    - If cannot determine → unified_id="oops"

In [ ]:

people_with_id = []
people_no_id = []

for data in multiples_dict.values():
    person_id = data["person_id"]
    unified_old = data["unified_id"]
    last = data["family_name"]
    given_combined = data["given_names"]

    given_split = given_combined.split("und")

    for i, name in enumerate(given_split):
        name_new = name.strip()
        name_new_norm = normalise_text(name).strip(". ")
        last_replace = last.replace("ö", "oe")
        last_norm = normalise_text(last).strip(". ").replace(" ", "-")
        unified_new = last_norm + "_" + name_new_norm

        if i == 0:
            people_with_id.append({
                **data,
                "unified_id": unified_new,
                "given_names": name_new
            })
        else:
            people_no_id.append({
                **data,
                "person_id": None,
                "unified_id": unified_new,
                "family_name": last,
                "given_names": name_new
            })

# rprint(f"count with id: {len(people_with_id)}")
# rprint(f"count no id: {len(people_no_id)}")
# rprint(people_with_id[:3])
# rprint(people_no_id[:3])

# with open("people_to_update.json", "w") as f:
#     json.dump(people_with_id, f, ensure_ascii=False, indent=2)
# with open("people_to_add.json", "w") as f:
#     json.dump(people_no_id, f, ensure_ascii=False, indent=2)
